# Create Refresh-Safe Operational Tables

## Purpose

This notebook creates a refresh-safe operational data model that separates source-controlled case information from agent-maintained updates.

The process:

- rebuilds `source_cases` from the latest source data
- creates agent update records only for new case IDs
- preserves existing agent notes and operational updates
- creates a combined `operational_case_view` for SQL and Power BI

This design prevents source refreshes from overwriting agent-maintained information.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

In [2]:
database_path = Path(
    "../sql/missed_refunds.db"
)

connection = sqlite3.connect(
    database_path
)

print(f"Connected to: {database_path}")

Connected to: ..\sql\missed_refunds.db


## Load Existing Source Data

The current `missed_refunds` table provides the case records used to build the refreshable source table.

In [3]:
source_query = """
SELECT *
FROM missed_refunds;
"""

existing_source_df = pd.read_sql_query(
    source_query,
    connection,
    parse_dates=["Snapshot Date"],
)

print(f"Rows loaded: {len(existing_source_df)}")
print(f"Columns loaded: {len(existing_source_df.columns)}")

existing_source_df.head()

Rows loaded: 2871
Columns loaded: 13


,Case ID,Policy Number,Client Number,Snapshot Date,Outstanding Amount,Cancellation Status,Cancelling Department,Cancelling Agent,Agent Working,Refund Type,Root Cause,Customer Contacted,Outcome
0,DH000001,PET000001,CL000029,2025-01-31,3.11,Cancelled,Customer Service,CS018,DH001,Cancellation Before Premium Due,Payment Date Misunderstood,Yes,Actioned
1,DH000002,PET000002,CL000155,2025-01-31,2.39,Cancelled,Customer Service,CS008,DH002,Cancellation Before Premium Due,Waiting for Information,Yes,No Action
2,DH000003,PET000003,CL000179,2025-01-31,21.12,Cancelled,Retentions,RET013,DH001,Death of Pet,Payment Date Misunderstood,Yes,Actioned
3,DH000004,PET000004,CL000032,2025-01-31,175.71,Cancelled,Claims,CLM010,DH002,Cancellation Before Premium Due,Waiting for Information,Yes,Actioned
4,DH000005,PET000005,CL000075,2025-01-31,271.06,Cancelled,Retentions,RET015,DH002,Death of Pet,Agent Forgot,No,Actioned


## Create Source Cases Table

The refreshable source table contains case attributes supplied by the source data. Agent-maintained fields are excluded.

In [4]:
source_case_columns = [
    "Case ID",
    "Policy Number",
    "Client Number",
    "Snapshot Date",
    "Outstanding Amount",
    "Cancellation Status",
    "Cancelling Department",
    "Cancelling Agent",
    "Refund Type",
    "Root Cause",
    "Customer Contacted",
]

source_cases_df = existing_source_df[
    source_case_columns
].copy()

source_cases_df.to_sql(
    name="source_cases",
    con=connection,
    if_exists="replace",
    index=False,
)

print(f"Created source_cases: {len(source_cases_df)} rows")

source_cases_df.head()

Created source_cases: 2871 rows


,Case ID,Policy Number,Client Number,Snapshot Date,Outstanding Amount,Cancellation Status,Cancelling Department,Cancelling Agent,Refund Type,Root Cause,Customer Contacted
0,DH000001,PET000001,CL000029,2025-01-31,3.11,Cancelled,Customer Service,CS018,Cancellation Before Premium Due,Payment Date Misunderstood,Yes
1,DH000002,PET000002,CL000155,2025-01-31,2.39,Cancelled,Customer Service,CS008,Cancellation Before Premium Due,Waiting for Information,Yes
2,DH000003,PET000003,CL000179,2025-01-31,21.12,Cancelled,Retentions,RET013,Death of Pet,Payment Date Misunderstood,Yes
3,DH000004,PET000004,CL000032,2025-01-31,175.71,Cancelled,Claims,CLM010,Cancellation Before Premium Due,Waiting for Information,Yes
4,DH000005,PET000005,CL000075,2025-01-31,271.06,Cancelled,Retentions,RET015,Death of Pet,Agent Forgot,No


## Create Persistent Agent Updates Table

The agent table is created only if it does not already exist.

Rerunning this notebook preserves existing notes, statuses and completion information.

In [5]:
create_agent_updates_query = """
CREATE TABLE IF NOT EXISTS agent_updates (
    "Case ID" TEXT PRIMARY KEY,
    "Agent Working" TEXT,
    "Agent Notes" TEXT,
    "Case Status" TEXT NOT NULL DEFAULT 'Open',
    "Final Outcome" TEXT,
    "Completed By" TEXT,
    "Completion Date" TEXT,
    "Last Updated Date" TEXT NOT NULL
);
"""

connection.execute(
    create_agent_updates_query
)

connection.commit()

print("agent_updates table is available.")

agent_updates table is available.


In [6]:
existing_agent_ids = pd.read_sql_query(
    """
    SELECT "Case ID"
    FROM agent_updates;
    """,
    connection,
)

existing_id_set = set(
    existing_agent_ids["Case ID"]
)

new_case_mask = ~source_cases_df[
    "Case ID"
].isin(existing_id_set)

new_source_cases = source_cases_df.loc[
    new_case_mask
].copy()

print(
    f"Existing agent records: "
    f"{len(existing_agent_ids)}"
)

print(
    f"New records to add: "
    f"{len(new_source_cases)}"
)

Existing agent records: 2871
New records to add: 0


## Add New Agent Update Records

Default agent records are created only for case IDs not already present in `agent_updates`.

In [7]:
initial_update_dates = (
    new_source_cases["Snapshot Date"]
    + pd.offsets.MonthBegin(1)
    + pd.Timedelta(days=14)
)

new_agent_updates_df = pd.DataFrame({
    "Case ID": new_source_cases["Case ID"],
    "Agent Working": pd.NA,
    "Agent Notes": pd.NA,
    "Case Status": "Open",
    "Final Outcome": pd.NA,
    "Completed By": pd.NA,
    "Completion Date": pd.NA,
    "Last Updated Date": (
        initial_update_dates
        .dt.strftime("%Y-%m-%d")
    ),
})

if not new_agent_updates_df.empty:
    new_agent_updates_df.to_sql(
        name="agent_updates",
        con=connection,
        if_exists="append",
        index=False,
    )

print(
    f"Agent records added: "
    f"{len(new_agent_updates_df)}"
)

Agent records added: 0


## Create Operational Case View

The reporting view joins refreshable source information to persistent agent updates using `Case ID`.

In [8]:
create_view_query = """
DROP VIEW IF EXISTS operational_case_view;

CREATE VIEW operational_case_view AS
SELECT
    source."Case ID",
    source."Policy Number",
    source."Client Number",
    source."Snapshot Date",
    source."Outstanding Amount",
    source."Cancellation Status",
    source."Cancelling Department",
    source."Cancelling Agent",
    source."Refund Type",
    source."Root Cause",
    source."Customer Contacted",
    updates."Agent Working",
    updates."Agent Notes",
    updates."Case Status",
    updates."Final Outcome",
    updates."Completed By",
    updates."Completion Date",
    updates."Last Updated Date"
FROM source_cases AS source
LEFT JOIN agent_updates AS updates
    ON source."Case ID" = updates."Case ID";
"""

connection.executescript(
    create_view_query
)

connection.commit()

print("Created view: operational_case_view")

Created view: operational_case_view


In [9]:
validation_query = """
SELECT
    (SELECT COUNT(*) FROM source_cases)
        AS source_rows,

    (SELECT COUNT(*) FROM agent_updates)
        AS agent_update_rows,

    (SELECT COUNT(*) FROM operational_case_view)
        AS view_rows,

    (
        SELECT COUNT(*)
        FROM operational_case_view
        WHERE "Case Status" IS NULL
    ) AS missing_agent_updates;
"""

pd.read_sql_query(
    validation_query,
    connection,
)

,source_rows,agent_update_rows,view_rows,missing_agent_updates
0,2871,2871,2871,0


In [10]:
operational_view_df = pd.read_sql_query(
    """
    SELECT *
    FROM operational_case_view;
    """,
    connection,
    parse_dates=[
        "Snapshot Date",
        "Completion Date",
        "Last Updated Date",
    ],
)

print(
    f"Operational view rows: "
    f"{len(operational_view_df)}"
)

print(
    f"Operational view columns: "
    f"{len(operational_view_df.columns)}"
)

operational_view_df.head()

Operational view rows: 2871
Operational view columns: 18


,Case ID,Policy Number,Client Number,Snapshot Date,Outstanding Amount,Cancellation Status,Cancelling Department,Cancelling Agent,Refund Type,Root Cause,Customer Contacted,Agent Working,Agent Notes,Case Status,Final Outcome,Completed By,Completion Date,Last Updated Date
0,DH000001,PET000001,CL000029,2025-01-31,3.11,Cancelled,Customer Service,CS018,Cancellation Before Premium Due,Payment Date Misunderstood,Yes,None,None,Open,None,None,NaT,2025-02-15
1,DH000002,PET000002,CL000155,2025-01-31,2.39,Cancelled,Customer Service,CS008,Cancellation Before Premium Due,Waiting for Information,Yes,None,None,Open,None,None,NaT,2025-02-15
2,DH000003,PET000003,CL000179,2025-01-31,21.12,Cancelled,Retentions,RET013,Death of Pet,Payment Date Misunderstood,Yes,None,None,Open,None,None,NaT,2025-02-15
3,DH000004,PET000004,CL000032,2025-01-31,175.71,Cancelled,Claims,CLM010,Cancellation Before Premium Due,Waiting for Information,Yes,None,None,Open,None,None,NaT,2025-02-15
4,DH000005,PET000005,CL000075,2025-01-31,271.06,Cancelled,Retentions,RET015,Death of Pet,Agent Forgot,No,None,None,Open,None,None,NaT,2025-02-15


## Validation Summary

The refresh-safe model was created successfully:

- `source_cases` contains 2,871 refreshable source records
- `agent_updates` contains one persistent record per Case ID
- rerunning the source refresh created zero duplicate agent records
- `operational_case_view` contains 2,871 joined rows and 18 columns
- every source case has a matching agent update record
- source refreshes replace only `source_cases`
- agent notes and operational updates remain stored separately in `agent_updates`

This design removes the risk of source-data refreshes overwriting agent-maintained notes and outcomes.

## Create Power Query Input Files

The refreshable source table and persistent agent-update table are saved separately for Power Query.

The source file is rebuilt during refresh. Existing agent records are preserved, with default rows appended only for new Case IDs.

In [15]:
connection = sqlite3.connect(database_path)

print("Database connection reopened.")

Database connection reopened.


In [16]:
operational_folder = Path(
    "../data/operational"
)

source_cases_output = (
    operational_folder
    / "source_cases.csv"
)

agent_updates_output = (
    operational_folder
    / "agent_updates.csv"
)

operational_folder.mkdir(
    parents=True,
    exist_ok=True,
)

source_cases_df.to_csv(
    source_cases_output,
    index=False,
    date_format="%Y-%m-%d",
)

database_agent_updates_df = (
    pd.read_sql_query(
        """
        SELECT *
        FROM agent_updates;
        """,
        connection,
    )
)

if agent_updates_output.exists():

    existing_agent_file_df = pd.read_csv(
        agent_updates_output,
        dtype={"Case ID": "string"},
    )

    new_file_records = (
        database_agent_updates_df[
            ~database_agent_updates_df[
                "Case ID"
            ].isin(
                existing_agent_file_df[
                    "Case ID"
                ]
            )
        ]
    )

    agent_updates_file_df = pd.concat(
        [
            existing_agent_file_df,
            new_file_records,
        ],
        ignore_index=True,
    )

else:
    agent_updates_file_df = (
        database_agent_updates_df.copy()
    )

agent_updates_file_df.to_csv(
    agent_updates_output,
    index=False,
)

print(f"Saved source file: {source_cases_output}")
print(
    f"Agent records preserved: "
    f"{len(agent_updates_file_df)}"
)

Saved source file: ..\data\operational\source_cases.csv
Agent records preserved: 2871


In [13]:
test_case_id = "DH000050"

test_mask = (
    agent_updates_file_df["Case ID"] == test_case_id
)

agent_updates_file_df.loc[
    test_mask,
    "Agent Working",
] = "DH001"

agent_updates_file_df.loc[
    test_mask,
    "Agent Notes",
] = "Refund requires senior confirmation before processing."

agent_updates_file_df.loc[
    test_mask,
    "Case Status",
] = "Awaiting Senior Review"

agent_updates_file_df.to_csv(
    agent_updates_output,
    index=False,
)

agent_updates_file_df.loc[test_mask]

,Case ID,Agent Working,Agent Notes,Case Status,Final Outcome,Completed By,Completion Date,Last Updated Date
49,DH000050,DH001,Refund requires senior confirmation before pro...,Awaiting Senior Review,None,None,None,2025-02-15


In [17]:
preservation_check_df = pd.read_csv(
    agent_updates_output,
    dtype={"Case ID": "string"},
)

preservation_check_df.loc[
    preservation_check_df["Case ID"] == "DH000050"
]

,Case ID,Agent Working,Agent Notes,Case Status,Final Outcome,Completed By,Completion Date,Last Updated Date
49,DH000050,DH001,Refund requires senior confirmation before pro...,Awaiting Senior Review,NaN,NaN,NaN,2025-02-15


In [12]:
connection.close()

print("Database connection closed.")

Database connection closed.
